In [0]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("test").getOrCreate()


df = spark.read.format("json")\
    .option("header","true")\
    .option("inferSchema","true")\
    .option("multiLine","true")\
    .load("/Volumes/workspace/default/dev_ecommerce")
df.show()

+--------------------+
|           ecommerce|
+--------------------+
|{[{Electronics, P...|
+--------------------+



In [0]:
from pyspark.sql.functions import current_timestamp, current_date, col

df = df.withColumn("ingestion_date", current_date())
df = df.withColumn("ingestion_timestamp", current_timestamp())
df = df.withColumn("source_file", col("_metadata.file_path"))


df.write.mode("append").parquet("/Volumes/workspace/default/dev_ecommerce/Bronze_layer")

In [0]:
df1 = spark.read.parquet(
    "/Volumes/workspace/default/dev_ecommerce/Bronze_layer"
)
df1.show()


+--------------------+--------------+--------------------+--------------------+
|           ecommerce|ingestion_date| ingestion_timestamp|         source_file|
+--------------------+--------------+--------------------+--------------------+
|{[{Electronics, P...|    2026-08-17|2026-08-17 16:17:...|dbfs:/Volumes/wor...|
+--------------------+--------------+--------------------+--------------------+



In [0]:
from pyspark.sql.functions import explode

df1=df1.select("*","ecommerce.*").drop("ecommerce").select("*",explode("orders").alias("order")).drop("orders").select("*","order.*").drop("order").select("*","delivery.*").drop("delivery").select("*","payment.*").drop("payment").show()


+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+------------+--------+-----+----------+---------------+--------+-----------+-----------+-----------+-------------+------------+----------------+------+----------+-------+--------------+
|ingestion_date| ingestion_timestamp|         source_file|       category|     city|customer_id|customer_name|discount|            email|  order_date|order_id|price|product_id|   product_name|quantity|      state|actual_date|delivery_id|expected_date|     partner|          status|amount|    method| status|transaction_id|
+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+------------+--------+-----+----------+---------------+--------+-----------+-----------+-----------+-------------+------------+----------------+------+----------+-------+--------------+
|    2026-08-17|2026-08-17 16:1

In [0]:

df1 = df1.dropDuplicates(["order_id"])
df1.show()

+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+------------+--------+-----+----------+---------------+--------+-----------+-----------+-----------+-------------+------------+----------------+------+----------+-------+--------------+
|ingestion_date| ingestion_timestamp|         source_file|       category|     city|customer_id|customer_name|discount|            email|  order_date|order_id|price|product_id|   product_name|quantity|      state|actual_date|delivery_id|expected_date|     partner|          status|amount|    method| status|transaction_id|
+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+------------+--------+-----+----------+---------------+--------+-----------+-----------+-----------+-------------+------------+----------------+------+----------+-------+--------------+
|    2026-08-17|2026-08-17 16:1

In [0]:
df1 = df1.fillna({
    "discount": 0,
    "quantity": 1
})

df1 = df1.dropna(subset=["order_id", "customer_id"])
df1.show()

+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+------------+--------+-----+----------+---------------+--------+-----------+-----------+-----------+-------------+---------+----------------+------+----------+-------+--------------+
|ingestion_date| ingestion_timestamp|         source_file|       category|     city|customer_id|customer_name|discount|            email|  order_date|order_id|price|product_id|   product_name|quantity|      state|actual_date|delivery_id|expected_date|  partner|          status|amount|    method| status|transaction_id|
+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+------------+--------+-----+----------+---------------+--------+-----------+-----------+-----------+-------------+---------+----------------+------+----------+-------+--------------+
|    2026-08-17|2026-08-17 16:17:...|dbf

In [0]:
from pyspark.sql.functions import col, expr

df1 = df1.withColumn("price", expr("try_cast(price as double)"))
df1 = df1.withColumn("quantity", expr("try_cast(quantity as int)"))
df1 = df1.withColumn("discount", expr("try_cast(discount as double)"))
df1.show()

+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+------------+--------+-------+----------+---------------+--------+-----------+-----------+-----------+-------------+---------+----------------+------+----------+-------+--------------+
|ingestion_date| ingestion_timestamp|         source_file|       category|     city|customer_id|customer_name|discount|            email|  order_date|order_id|  price|product_id|   product_name|quantity|      state|actual_date|delivery_id|expected_date|  partner|          status|amount|    method| status|transaction_id|
+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+------------+--------+-------+----------+---------------+--------+-----------+-----------+-----------+-------------+---------+----------------+------+----------+-------+--------------+
|    2026-08-17|2026-08-17 16:17:.

In [0]:
for c in df.columns:
    df1 = df1.withColumnRenamed(c, c.lower().strip().replace(" ", "_"))
df1.show()    

+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+------------+--------+-------+----------+---------------+--------+-----------+-----------+-----------+-------------+---------+----------------+------+----------+-------+--------------+
|ingestion_date| ingestion_timestamp|         source_file|       category|     city|customer_id|customer_name|discount|            email|  order_date|order_id|  price|product_id|   product_name|quantity|      state|actual_date|delivery_id|expected_date|  partner|          status|amount|    method| status|transaction_id|
+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+------------+--------+-------+----------+---------------+--------+-----------+-----------+-----------+-------------+---------+----------------+------+----------+-------+--------------+
|    2026-08-17|2026-08-17 16:17:.

In [0]:
df1 = df1.filter(col("price") >= 0)
df1 = df1.filter(col("quantity") > 0)

df1.show()

+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+----------+--------+-------+----------+--------------+--------+-----------+-----------+-----------+-------------+---------+----------------+------+----------+-------+--------------+
|ingestion_date| ingestion_timestamp|         source_file|       category|     city|customer_id|customer_name|discount|            email|order_date|order_id|  price|product_id|  product_name|quantity|      state|actual_date|delivery_id|expected_date|  partner|          status|amount|    method| status|transaction_id|
+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+----------+--------+-------+----------+--------------+--------+-----------+-----------+-----------+-------------+---------+----------------+------+----------+-------+--------------+
|    2026-08-17|2026-08-17 16:17:...|dbfs:/

In [0]:
from pyspark.sql.functions import to_date, coalesce, expr


df1 = df1.withColumn(
    "order_date",
    coalesce(
        expr("try_to_date(order_date, 'yyyy-MM-dd')"),
        expr("try_to_date(order_date, 'dd/MM/yyyy')"),
        expr("try_to_date(order_date, 'yyyy/MM/dd')")
    )
)

df1.show()

+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+----------+--------+-------+----------+--------------+--------+-----------+-----------+-----------+-------------+---------+----------------+------+----------+-------+--------------+
|ingestion_date| ingestion_timestamp|         source_file|       category|     city|customer_id|customer_name|discount|            email|order_date|order_id|  price|product_id|  product_name|quantity|      state|actual_date|delivery_id|expected_date|  partner|          status|amount|    method| status|transaction_id|
+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+----------+--------+-------+----------+--------------+--------+-----------+-----------+-----------+-------------+---------+----------------+------+----------+-------+--------------+
|    2026-08-17|2026-08-17 16:17:...|dbfs:/

In [0]:
df1 = df1.withColumn(
    "net_amount",
    (col("price") * col("quantity")) - col("discount")
)

df1.show()

+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+----------+--------+-------+----------+--------------+--------+-----------+-----------+-----------+-------------+---------+----------------+------+----------+-------+--------------+----------+
|ingestion_date| ingestion_timestamp|         source_file|       category|     city|customer_id|customer_name|discount|            email|order_date|order_id|  price|product_id|  product_name|quantity|      state|actual_date|delivery_id|expected_date|  partner|          status|amount|    method| status|transaction_id|net_amount|
+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+----------+--------+-------+----------+--------------+--------+-----------+-----------+-----------+-------------+---------+----------------+------+----------+-------+--------------+----------+
|    2026-

In [0]:
cols = list(df1.columns)

cols[20] = "delivery_status"
cols[23] = "payment_status"

df1_clean = df1.toDF(*cols)

df1_clean.write.mode("overwrite").parquet("/Volumes/workspace/default/dev_ecommerce/Silver_layer")


In [0]:

df_clean = spark.read.parquet("/Volumes/workspace/default/dev_ecommerce/Silver_layer")

df_clean.show()


+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+----------+--------+-------+----------+--------------+--------+-----------+-----------+-----------+-------------+---------+----------------+------+----------+--------------+--------------+----------+
|ingestion_date| ingestion_timestamp|         source_file|       category|     city|customer_id|customer_name|discount|            email|order_date|order_id|  price|product_id|  product_name|quantity|      state|actual_date|delivery_id|expected_date|  partner| delivery_status|amount|    method|payment_status|transaction_id|net_amount|
+--------------+--------------------+--------------------+---------------+---------+-----------+-------------+--------+-----------------+----------+--------+-------+----------+--------------+--------+-----------+-----------+-----------+-------------+---------+----------------+------+----------+--------------+--------------+-

In [0]:
from pyspark.sql.functions import *

daily_sales = df_clean.groupBy("order_date").agg(countDistinct("order_id").alias("total_orders"),
    sum("net_amount").alias("total_sales"),
    avg("net_amount").alias("average_order_value"),
    countDistinct("customer_id").alias("unique_customers")
).withColumnRenamed("order_date", "sale_date")

daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_daily_sales")

In [0]:
customer_analytics = df_clean.groupBy("customer_id", "customer_name").agg(
    countDistinct("order_id").alias("total_orders"),
    sum("net_amount").alias("total_spent"),
    avg("net_amount").alias("average_order_value"),
    max("order_date").alias("last_order_date")
)

customer_analytics = customer_analytics.withColumn(
    "customer_segment",
    when(col("total_spent") >= 100000, "Platinum")
    .when(col("total_spent") >= 50000, "Gold")
    .when(col("total_spent") >= 10000, "Silver")
    .otherwise("Bronze")
)

customer_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_customer_analytics")

In [0]:
product_analytics = df_clean.groupBy("product_id","product_name","category").agg(
    sum("quantity").alias("total_quantity_sold"),
    sum("net_amount").alias("total_revenue"),
    avg("price").alias("average_price")
)

product_analytics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_product_analytics")

In [0]:
from pyspark.sql.functions import datediff, col, when

delivery_performance = df_clean.select("delivery_id", "order_id","actual_date","expected_date","delivery_status"
)

delivery_performance = delivery_performance.withColumn("delay_days",
    datediff("actual_date", "expected_date")
)

delivery_performance = delivery_performance.withColumn("delivery_classification",
    when(col("delay_days") <= 0, "On Time")
    .when(col("delay_days") <= 3, "Slightly Delayed")
    .otherwise("Delayed")
)

delivery_performance.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.gold_delivery_performance")